# 从零实现 Conformer-CTC：语音编码、对齐动态规划与流式边界

本 Notebook 不调用 `nn.MultiheadAttention`、`nn.Transformer` 或 `nn.CTCLoss`。我们只用基础 PyTorch 层手写：

- Conformer 的 macaron FFN、自注意力与 depthwise convolution；
- padding mask、位置编码和 `ConformerCTC.forward`；
- log-space CTC 前向动态规划与 greedy collapse；
- 受控声学特征训练、对齐穷举 oracle、padding 不变性；
- 模型、词表、前端与训练快照绑定的可信推理接口。

输入是合成的帧级声学特征，不是真实语音。它能验证数学和接口，不能代表真实 ASR 的字错率、口音鲁棒性或流式延迟。

## 1. 运行合同

全程 CPU、单线程、固定种子，不下载音频或权重。训练与推理使用同一 feature dimension；真实系统还必须绑定采样率、窗长、步长、mel bins、归一化和 tokenizer 版本。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from copy import deepcopy  # 导入本单元所需的依赖。
from hashlib import sha256  # 导入本单元所需的依赖。
import itertools  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 4001  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
np.random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.__version__  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "seed": SEED, "device": str(DEVICE)})  # 执行当前语句以推进本节示例。

## 2. 受控可变长声学数据

词表 `0` 是 CTC blank，四个非 blank token 各有一个稳定原型。每个标签持续 2–3 帧并叠加小噪声，所以模型必须把帧序列映射为更短的标签序列。相邻 target 故意不重复；真实 CTC 若要表达连续相同标签，中间必须出现 blank。

`lengths` 是唯一可信的有效区域。padding 的具体数值不得影响有效帧输出，训练 loss 也不能读 padding。

In [ ]:
VOCAB = {0: "<blank>", 1: "上", 2: "下", 3: "左", 4: "右"}  # 计算并保存当前步骤的中间状态。
BLANK_ID, FEATURE_DIM = 0, 8  # 计算并保存当前步骤的中间状态。
prototypes = torch.zeros(len(VOCAB), FEATURE_DIM)  # 计算并保存当前步骤的中间状态。
for token in range(1, len(VOCAB)):  # 遍历输入元素以累积或检查结果。
    prototypes[token, token - 1] = 2.5  # 计算并保存当前步骤的中间状态。
    prototypes[token, 4 + (token % 4)] = 1.0  # 计算并保存当前步骤的中间状态。

def make_acoustic_batch(target_lists, seed):  # 定义本节可复用的核心函数。
    generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
    sequences = []  # 计算并保存当前步骤的中间状态。
    for row, target in enumerate(target_lists):  # 遍历输入元素以累积或检查结果。
        frames = []  # 计算并保存当前步骤的中间状态。
        for position, token in enumerate(target):  # 遍历输入元素以累积或检查结果。
            repeat = 2 + (row % 2)  # 计算并保存当前步骤的中间状态。
            base = prototypes[token].repeat(repeat, 1)  # 计算并保存当前步骤的中间状态。
            frames.append(base + 0.035 * torch.randn(base.shape, generator=generator))  # 计算并保存当前步骤的中间状态。
        sequences.append(torch.cat(frames))  # 执行当前语句以推进本节示例。
    lengths = torch.tensor([len(x) for x in sequences], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    padded = torch.zeros(len(sequences), int(lengths.max()), FEATURE_DIM)  # 计算并保存当前步骤的中间状态。
    for i, value in enumerate(sequences):  # 遍历输入元素以累积或检查结果。
        padded[i, : len(value)] = value  # 计算并保存当前步骤的中间状态。
    targets = [torch.tensor(x, dtype=torch.long) for x in target_lists]  # 计算并保存当前步骤的中间状态。
    return padded, lengths, targets  # 返回当前分支计算出的结果。

train_target_lists = [[1,2], [2,3], [3,4], [4,1], [1,3], [2,4], [3,1], [4,2]]  # 计算并保存当前步骤的中间状态。
valid_target_lists = [[1,4], [2,1], [3,2], [4,3]]  # 计算并保存当前步骤的中间状态。
train_features, train_lengths, train_targets = make_acoustic_batch(train_target_lists, SEED)  # 计算并保存当前步骤的中间状态。
valid_features, valid_lengths, valid_targets = make_acoustic_batch(valid_target_lists, SEED + 1)  # 计算并保存当前步骤的中间状态。

assert train_features.shape[0] == len(train_targets) == 8  # 用受控断言验证关键不变量。
assert int(train_lengths.min()) >= 4  # 用受控断言验证关键不变量。
assert all((target > BLANK_ID).all() for target in train_targets)  # 用受控断言验证关键不变量。
assert all(not bool((target[1:] == target[:-1]).any()) for target in train_targets)  # 用受控断言验证关键不变量。
print({"train_shape": tuple(train_features.shape), "lengths": train_lengths.tolist()})  # 执行当前语句以推进本节示例。

## 3. 手写 scaled multi-head self-attention

对每个 head，注意力为 `softmax(QK^T / sqrt(d_head))V`。key padding 在 softmax 前设为负无穷；query padding 在输出后清零。只检查 shape 不足以发现漏掉缩放，因此下面用恒等 Q/K/V 投影构造可手算数值 oracle。

In [ ]:
def length_mask(lengths, max_length=None):  # 定义本节可复用的核心函数。
    if lengths.ndim != 1 or lengths.dtype != torch.long or len(lengths) == 0 or bool((lengths <= 0).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("lengths_must_be_positive_vector")  # 遇到非法合同立即显式失败。
    width = int(lengths.max()) if max_length is None else int(max_length)  # 计算并保存当前步骤的中间状态。
    if bool((lengths > width).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("length_exceeds_tensor")  # 遇到非法合同立即显式失败。
    return torch.arange(width, device=lengths.device)[None, :] < lengths[:, None]  # 返回当前分支计算出的结果。

class ManualSelfAttention(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, heads):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if dim % heads != 0:  # 按当前条件选择后续控制路径。
            raise ValueError("dim_must_divide_heads")  # 遇到非法合同立即显式失败。
        self.dim, self.heads, self.head_dim = dim, heads, dim // heads  # 计算并保存当前步骤的中间状态。
        self.scale = self.head_dim ** -0.5  # 计算并保存当前步骤的中间状态。
        self.qkv = nn.Linear(dim, 3 * dim)  # 计算并保存当前步骤的中间状态。
        self.output = nn.Linear(dim, dim)  # 计算并保存当前步骤的中间状态。

    def forward(self, x, valid_mask):  # 定义本节可复用的核心函数。
        if x.ndim != 3 or valid_mask.shape != x.shape[:2] or not bool(valid_mask.any(dim=1).all()):  # 按当前条件选择后续控制路径。
            raise ValueError("attention_shape_or_empty_sequence")  # 遇到非法合同立即显式失败。
        batch, steps, _ = x.shape  # 计算并保存当前步骤的中间状态。
        qkv = self.qkv(x).view(batch, steps, 3, self.heads, self.head_dim)  # 计算并保存当前步骤的中间状态。
        q, k, v = qkv.unbind(dim=2)  # 计算并保存当前步骤的中间状态。
        q, k, v = (value.transpose(1, 2) for value in (q, k, v))  # 计算并保存当前步骤的中间状态。
        scores = (q @ k.transpose(-2, -1)) * self.scale  # 计算并保存当前步骤的中间状态。
        scores = scores.masked_fill(~valid_mask[:, None, None, :], float("-inf"))  # 计算并保存当前步骤的中间状态。
        weights = torch.softmax(scores, dim=-1)  # 计算并保存当前步骤的中间状态。
        context = (weights @ v).transpose(1, 2).contiguous().view(batch, steps, self.dim)  # 计算并保存当前步骤的中间状态。
        result = self.output(context) * valid_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return result, weights  # 返回当前分支计算出的结果。

oracle_attention = ManualSelfAttention(4, 2)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    oracle_attention.qkv.weight.copy_(torch.cat([torch.eye(4)] * 3))  # 执行当前语句以推进本节示例。
    oracle_attention.qkv.bias.zero_()  # 执行当前语句以推进本节示例。
    oracle_attention.output.weight.copy_(torch.eye(4))  # 执行当前语句以推进本节示例。
    oracle_attention.output.bias.zero_()  # 执行当前语句以推进本节示例。
oracle_x = torch.tensor([[[1., 0., 1., 0.], [0., 1., 0., 2.]]])  # 计算并保存当前步骤的中间状态。
oracle_mask = torch.tensor([[True, True]])  # 计算并保存当前步骤的中间状态。
oracle_out, oracle_weights = oracle_attention(oracle_x, oracle_mask)  # 计算并保存当前步骤的中间状态。
q = oracle_x.view(1, 2, 2, 2).transpose(1, 2)  # 计算并保存当前步骤的中间状态。
expected_weights = torch.softmax((q @ q.transpose(-2, -1)) / math.sqrt(2), dim=-1)  # 计算并保存当前步骤的中间状态。
expected_out = (expected_weights @ q).transpose(1, 2).reshape(1, 2, 4)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(oracle_weights, expected_weights, atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(oracle_out, expected_out, atol=1e-6)  # 用受控断言验证关键不变量。
assert not torch.allclose(expected_weights, torch.softmax(q @ q.transpose(-2, -1), dim=-1))  # 用受控断言验证关键不变量。

## 4. Conformer block 与 CTC head

一个 block 按 `1/2 FFN -> MHSA -> Conv -> 1/2 FFN -> LayerNorm` 排列。卷积分支用 pointwise GLU、depthwise temporal convolution 和逐时刻 LayerNorm，同时把 padding 在每个残差后重新清零。

这里不做时间下采样，故 shape 流为声学特征 `[B,T,F] -> [B,T,D] -> [B,T,|V|]`，有效长度始终是 `[B]`；attention 权重是 `[B,H,T,T]`。生产 Conformer 常用 convolutional subsampling，必须用精确公式同步更新 CTC lengths。

In [ ]:
def sinusoidal_positions(length, dim, device):  # 定义本节可复用的核心函数。
    position = torch.arange(length, device=device, dtype=torch.float32)[:, None]  # 计算并保存当前步骤的中间状态。
    frequency = torch.exp(torch.arange(0, dim, 2, device=device) * (-math.log(10000.0) / dim))  # 计算并保存当前步骤的中间状态。
    table = torch.zeros(length, dim, device=device)  # 计算并保存当前步骤的中间状态。
    table[:, 0::2] = torch.sin(position * frequency)  # 计算并保存当前步骤的中间状态。
    table[:, 1::2] = torch.cos(position * frequency[: table[:, 1::2].shape[1]])  # 计算并保存当前步骤的中间状态。
    return table  # 返回当前分支计算出的结果。

class FeedForwardModule(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, expansion=2):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.norm = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.first = nn.Linear(dim, expansion * dim)  # 计算并保存当前步骤的中间状态。
        self.second = nn.Linear(expansion * dim, dim)  # 计算并保存当前步骤的中间状态。
    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.second(F.silu(self.first(self.norm(x))))  # 返回当前分支计算出的结果。

class ConformerConvModule(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, kernel_size=5):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if kernel_size % 2 == 0:  # 按当前条件选择后续控制路径。
            raise ValueError("kernel_size_must_be_odd")  # 遇到非法合同立即显式失败。
        self.pointwise_in = nn.Conv1d(dim, 2 * dim, 1)  # 计算并保存当前步骤的中间状态。
        self.depthwise = nn.Conv1d(dim, dim, kernel_size, padding=kernel_size // 2, groups=dim)  # 计算并保存当前步骤的中间状态。
        self.norm = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.pointwise_out = nn.Conv1d(dim, dim, 1)  # 计算并保存当前步骤的中间状态。
    def forward(self, x, valid_mask):  # 定义本节可复用的核心函数。
        channel_mask = valid_mask[:, None, :]  # 计算并保存当前步骤的中间状态。
        y = self.pointwise_in((x * valid_mask.unsqueeze(-1)).transpose(1, 2))  # 计算并保存当前步骤的中间状态。
        y = F.glu(y, dim=1)  # 计算并保存当前步骤的中间状态。
        y = y * channel_mask  # 计算并保存当前步骤的中间状态。
        y = self.depthwise(y) * channel_mask  # 计算并保存当前步骤的中间状态。
        y = y.transpose(1, 2)  # 计算并保存当前步骤的中间状态。
        y = F.silu(self.norm(y)) * valid_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        y = y.transpose(1, 2)  # 计算并保存当前步骤的中间状态。
        return self.pointwise_out(y).transpose(1, 2) * valid_mask.unsqueeze(-1)  # 返回当前分支计算出的结果。

class ConformerBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, heads):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.ffn1, self.ffn2 = FeedForwardModule(dim), FeedForwardModule(dim)  # 计算并保存当前步骤的中间状态。
        self.attention_norm = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.attention = ManualSelfAttention(dim, heads)  # 计算并保存当前步骤的中间状态。
        self.conv_norm = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.convolution = ConformerConvModule(dim)  # 计算并保存当前步骤的中间状态。
        self.final_norm = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
    def forward(self, x, valid_mask):  # 定义本节可复用的核心函数。
        mask = valid_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        x = (x + 0.5 * self.ffn1(x)) * mask  # 计算并保存当前步骤的中间状态。
        attended, weights = self.attention(self.attention_norm(x), valid_mask)  # 计算并保存当前步骤的中间状态。
        x = (x + attended) * mask  # 计算并保存当前步骤的中间状态。
        x = (x + self.convolution(self.conv_norm(x), valid_mask)) * mask  # 计算并保存当前步骤的中间状态。
        x = self.final_norm(x + 0.5 * self.ffn2(x)) * mask  # 计算并保存当前步骤的中间状态。
        return x, weights  # 返回当前分支计算出的结果。

class ConformerCTC(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, feature_dim, vocab_size, dim=16, heads=2, layers=1):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.feature_dim, self.vocab_size = feature_dim, vocab_size  # 计算并保存当前步骤的中间状态。
        self.dim, self.heads, self.layers_count = dim, heads, layers  # 计算并保存当前步骤的中间状态。
        self.input_projection = nn.Linear(feature_dim, dim)  # 计算并保存当前步骤的中间状态。
        self.blocks = nn.ModuleList([ConformerBlock(dim, heads) for _ in range(layers)])  # 计算并保存当前步骤的中间状态。
        self.output_norm = nn.LayerNorm(dim)  # 计算并保存当前步骤的中间状态。
        self.ctc_head = nn.Linear(dim, vocab_size)  # 计算并保存当前步骤的中间状态。
    def forward(self, features, lengths):  # 定义本节可复用的核心函数。
        if features.ndim != 3 or features.shape[-1] != self.feature_dim or lengths.shape != (features.shape[0],) or lengths.device != features.device:  # 按当前条件选择后续控制路径。
            raise ValueError("conformer_input_contract")  # 遇到非法合同立即显式失败。
        if not torch.isfinite(features).all():  # 按当前条件选择后续控制路径。
            raise ValueError("nonfinite_features")  # 遇到非法合同立即显式失败。
        valid = length_mask(lengths, features.shape[1])  # 计算并保存当前步骤的中间状态。
        x = self.input_projection(features * valid.unsqueeze(-1))  # 计算并保存当前步骤的中间状态。
        x = (x + sinusoidal_positions(features.shape[1], self.dim, features.device)) * valid.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        attention_maps = []  # 计算并保存当前步骤的中间状态。
        for block in self.blocks:  # 遍历输入元素以累积或检查结果。
            x, weights = block(x, valid)  # 计算并保存当前步骤的中间状态。
            attention_maps.append(weights)  # 执行当前语句以推进本节示例。
        logits = self.ctc_head(self.output_norm(x)) * valid.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return F.log_softmax(logits, dim=-1), valid, attention_maps  # 返回当前分支计算出的结果。

model40 = ConformerCTC(FEATURE_DIM, len(VOCAB)).to(DEVICE)  # 计算并保存当前步骤的中间状态。
probe_log_probs, probe_valid, probe_attention = model40(train_features[:2], train_lengths[:2])  # 计算并保存当前步骤的中间状态。
assert probe_log_probs.shape == (2, train_features.shape[1], len(VOCAB))  # 用受控断言验证关键不变量。
assert probe_attention[0].shape[:2] == (2, 2)  # 用受控断言验证关键不变量。
assert torch.allclose(probe_log_probs.exp().sum(-1), torch.ones_like(probe_valid, dtype=torch.float32), atol=1e-6)  # 用受控断言验证关键不变量。

## 5. 从零实现 CTC log-space 动态规划

给 target `y[U]` 插入 blank 得到扩展序列 `z[S]`，其中 `S=2U+1`；输入 log-probability 为 `[T,C]`，动态规划表 `alpha` 为 `[T,S]`。状态可从自身、前一状态转移；当前标签既非 blank 又不同于隔一个状态时，还可跳两格。

递推可写为 `alpha[t,s] = log p_t(z_s) + logsumexp(alpha[t-1,s], alpha[t-1,s-1], skip)`；只有 `z_s != blank` 且 `z_s != z_{s-2}` 时才包含 `skip=alpha[t-1,s-2]`。终值是最后 blank 与最后标签两个状态的 `logsumexp`。下面把长度 3、词表 3 的所有 frame alignment 穷举，与动态规划对照。这个 oracle 会抓到“先删 blank 还是先合并重复”、跳转条件和 final state 取错等问题。

In [ ]:
def ctc_log_probability(log_probs, target, blank_id=0):  # 定义本节可复用的核心函数。
    if log_probs.ndim != 2 or target.ndim != 1 or len(target) == 0:  # 按当前条件选择后续控制路径。
        raise ValueError("ctc_single_example_shape")  # 遇到非法合同立即显式失败。
    if bool((target == blank_id).any()) or bool((target < 0).any()) or bool((target >= log_probs.shape[1]).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("invalid_ctc_target")  # 遇到非法合同立即显式失败。
    extended = torch.full((2 * len(target) + 1,), blank_id, dtype=torch.long, device=target.device)  # 计算并保存当前步骤的中间状态。
    extended[1::2] = target  # 计算并保存当前步骤的中间状态。
    adjacent_repeats = int((target[1:] == target[:-1]).sum()) if len(target) > 1 else 0  # 计算并保存当前步骤的中间状态。
    minimum_steps = len(target) + adjacent_repeats  # 计算并保存当前步骤的中间状态。
    if log_probs.shape[0] < minimum_steps:  # 按当前条件选择后续控制路径。
        raise ValueError("insufficient_input_steps")  # 遇到非法合同立即显式失败。
    negative_inf = log_probs.new_tensor(float("-inf"))  # 计算并保存当前步骤的中间状态。
    alpha = [log_probs[0, extended[0]]]  # 计算并保存当前步骤的中间状态。
    for state in range(1, len(extended)):  # 遍历输入元素以累积或检查结果。
        alpha.append(log_probs[0, extended[state]] if state == 1 else negative_inf)  # 计算并保存当前步骤的中间状态。
    alpha = torch.stack(alpha)  # 计算并保存当前步骤的中间状态。
    for time in range(1, log_probs.shape[0]):  # 遍历输入元素以累积或检查结果。
        next_alpha = []  # 计算并保存当前步骤的中间状态。
        for state, symbol in enumerate(extended.tolist()):  # 遍历输入元素以累积或检查结果。
            candidates = [alpha[state]]  # 计算并保存当前步骤的中间状态。
            if state >= 1:  # 按当前条件选择后续控制路径。
                candidates.append(alpha[state - 1])  # 执行当前语句以推进本节示例。
            if state >= 2 and symbol != blank_id and symbol != int(extended[state - 2]):  # 按当前条件选择后续控制路径。
                candidates.append(alpha[state - 2])  # 执行当前语句以推进本节示例。
            next_alpha.append(torch.logsumexp(torch.stack(candidates), dim=0) + log_probs[time, symbol])  # 计算并保存当前步骤的中间状态。
        alpha = torch.stack(next_alpha)  # 计算并保存当前步骤的中间状态。
    return torch.logsumexp(alpha[-2:], dim=0)  # 返回当前分支计算出的结果。

def ctc_loss_batch(log_probs, lengths, targets, blank_id=0):  # 定义本节可复用的核心函数。
    if lengths.dtype != torch.long or lengths.shape != (log_probs.shape[0],) or bool((lengths > log_probs.shape[1]).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("ctc_length_contract")  # 遇到非法合同立即显式失败。
    if len(targets) != log_probs.shape[0]:  # 按当前条件选择后续控制路径。
        raise ValueError("target_batch_mismatch")  # 遇到非法合同立即显式失败。
    values = [  # 计算并保存当前步骤的中间状态。
        -ctc_log_probability(log_probs[i, : int(lengths[i])], targets[i].to(log_probs.device), blank_id)  # 执行当前语句以推进本节示例。
        for i in range(log_probs.shape[0])  # 遍历输入元素以累积或检查结果。
    ]  # 执行当前语句以推进本节示例。
    return torch.stack(values).mean()  # 返回当前分支计算出的结果。

def ctc_collapse(path, blank_id=0):  # 定义本节可复用的核心函数。
    merged = [value for index, value in enumerate(path) if index == 0 or value != path[index - 1]]  # 计算并保存当前步骤的中间状态。
    return [value for value in merged if value != blank_id]  # 返回当前分支计算出的结果。

tiny_prob = torch.tensor([[0.5,0.4,0.1], [0.2,0.7,0.1], [0.6,0.3,0.1]], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
tiny_log = tiny_prob.log()  # 计算并保存当前步骤的中间状态。
target_one = torch.tensor([1])  # 计算并保存当前步骤的中间状态。
brute_probability = sum(  # 计算并保存当前步骤的中间状态。
    math.prod(float(tiny_prob[t, symbol]) for t, symbol in enumerate(path))  # 执行当前语句以推进本节示例。
    for path in itertools.product(range(3), repeat=3) if ctc_collapse(path) == [1]  # 遍历输入元素以累积或检查结果。
)  # 执行当前语句以推进本节示例。
dp_probability = float(ctc_log_probability(tiny_log, target_one).exp())  # 计算并保存当前步骤的中间状态。
assert math.isclose(dp_probability, brute_probability, rel_tol=1e-10, abs_tol=1e-12)  # 用受控断言验证关键不变量。
assert ctc_collapse([0,1,1,0,2,2]) == [1,2]  # 用受控断言验证关键不变量。
assert ctc_collapse([1,0,1]) == [1,1]  # 用受控断言验证关键不变量。

repeat_target40 = torch.tensor([1, 1])  # 计算并保存当前步骤的中间状态。
repeat_probability40 = sum(  # 计算并保存当前步骤的中间状态。
    math.prod(float(tiny_prob[t, symbol]) for t, symbol in enumerate(path))  # 执行当前语句以推进本节示例。
    for path in itertools.product(range(3), repeat=3) if ctc_collapse(path) == [1, 1]  # 遍历输入元素以累积或检查结果。
)  # 执行当前语句以推进本节示例。
repeat_dp40 = float(ctc_log_probability(tiny_log, repeat_target40).exp())  # 计算并保存当前步骤的中间状态。
assert repeat_probability40 > 0  # 用受控断言验证关键不变量。
assert math.isclose(repeat_dp40, repeat_probability40, rel_tol=1e-10, abs_tol=1e-12)  # 用受控断言验证关键不变量。

## 6. 受控训练与 greedy decoding

训练只读取 train batch；validation 不参与梯度或 step 数选择。目标是证明自定义 attention、卷积和 CTC DP 可以共同反传。真实 ASR 要按说话人/录音分组切分，并报告 CER/WER、长音频、噪声、口音和实时率。

In [ ]:
def greedy_ctc_decode(log_probs, lengths, blank_id=0):  # 定义本节可复用的核心函数。
    best = log_probs.argmax(dim=-1)  # 计算并保存当前步骤的中间状态。
    return [ctc_collapse(best[i, : int(lengths[i])].tolist(), blank_id) for i in range(len(lengths))]  # 返回当前分支计算出的结果。

torch.manual_seed(SEED + 2)  # 执行当前语句以推进本节示例。
model40 = ConformerCTC(FEATURE_DIM, len(VOCAB), dim=16, heads=2, layers=1)  # 计算并保存当前步骤的中间状态。
optimizer40 = torch.optim.Adam(model40.parameters(), lr=0.018)  # 计算并保存当前步骤的中间状态。
initial_state40 = {name: value.detach().clone() for name, value in model40.state_dict().items()}  # 计算并保存当前步骤的中间状态。
losses40 = []  # 计算并保存当前步骤的中间状态。
for step in range(61):  # 遍历输入元素以累积或检查结果。
    model40.train()  # 执行当前语句以推进本节示例。
    optimizer40.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    log_probs, _, _ = model40(train_features, train_lengths)  # 计算并保存当前步骤的中间状态。
    loss = ctc_loss_batch(log_probs, train_lengths, train_targets, BLANK_ID)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    if step == 0:  # 按当前条件选择后续控制路径。
        first_grad40 = torch.sqrt(sum((p.grad ** 2).sum() for p in model40.parameters() if p.grad is not None))  # 计算并保存当前步骤的中间状态。
    torch.nn.utils.clip_grad_norm_(model40.parameters(), 5.0)  # 执行当前语句以推进本节示例。
    optimizer40.step()  # 执行当前语句以推进本节示例。
    losses40.append(float(loss.detach()))  # 执行当前语句以推进本节示例。

model40.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    valid_log_probs, _, _ = model40(valid_features, valid_lengths)  # 计算并保存当前步骤的中间状态。
valid_predictions = greedy_ctc_decode(valid_log_probs, valid_lengths)  # 计算并保存当前步骤的中间状态。
sequence_accuracy40 = np.mean([pred == gold for pred, gold in zip(valid_predictions, valid_target_lists)])  # 计算并保存当前步骤的中间状态。
parameters_changed40 = any(not torch.equal(initial_state40[name], value) for name, value in model40.state_dict().items())  # 计算并保存当前步骤的中间状态。
print({"loss": [round(losses40[0],4), round(losses40[-1],4)], "validation_sequence_accuracy": sequence_accuracy40})  # 执行当前语句以推进本节示例。
assert first_grad40 > 0 and torch.isfinite(first_grad40)  # 用受控断言验证关键不变量。
assert parameters_changed40  # 用受控断言验证关键不变量。
assert losses40[-1] < 0.25 * losses40[0]  # 用受控断言验证关键不变量。
assert sequence_accuracy40 >= 0.75  # 用受控断言验证关键不变量。

## 7. padding、未来垃圾帧与错误输入

把有效序列后面接任意垃圾帧但保持 `length` 不变，有效 logits 必须相同。这个测试覆盖 attention key mask、query 清零和卷积边界。全空长度、非有限输入、错误 feature dimension 都应 fail closed。

In [ ]:
model40.eval()  # 执行当前语句以推进本节示例。
single_length = valid_lengths[:1]  # 计算并保存当前步骤的中间状态。
single = valid_features[:1, : int(single_length[0])]  # 计算并保存当前步骤的中间状态。
garbage = torch.randn(1, 4, FEATURE_DIM) * 100  # 计算并保存当前步骤的中间状态。
extended = torch.cat([single, garbage], dim=1)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    base_logits = model40(single, single_length)[0]  # 计算并保存当前步骤的中间状态。
    extended_logits = model40(extended, single_length)[0][:, : single.shape[1]]  # 计算并保存当前步骤的中间状态。
assert torch.allclose(base_logits, extended_logits, atol=2e-5)  # 用受控断言验证关键不变量。

def rejected40(fn, expected=(ValueError, RuntimeError, TypeError)):  # 定义本节可复用的核心函数。
    try:  # 尝试执行可能失败的受控操作。
        fn()  # 执行当前语句以推进本节示例。
        return False  # 返回当前分支计算出的结果。
    except expected:  # 捕获预期异常并验证失败分支。
        return True  # 返回当前分支计算出的结果。

assert rejected40(lambda: model40(torch.zeros(1,2,FEATURE_DIM), torch.tensor([0])))  # 用受控断言验证关键不变量。
assert rejected40(lambda: model40(torch.zeros(1,3,FEATURE_DIM), torch.tensor([2.5])))  # 用受控断言验证关键不变量。
assert rejected40(lambda: model40(torch.zeros(1,2,FEATURE_DIM + 1), torch.tensor([2])))  # 用受控断言验证关键不变量。
bad_features40 = torch.zeros(1,2,FEATURE_DIM); bad_features40[0,0,0] = float("nan")  # 计算并保存当前步骤的中间状态。
assert rejected40(lambda: model40(bad_features40, torch.tensor([2])))  # 用受控断言验证关键不变量。
assert rejected40(lambda: ctc_log_probability(torch.randn(1,5), torch.tensor([1,2])))  # 用受控断言验证关键不变量。
assert rejected40(lambda: ctc_log_probability(torch.randn(2,5), torch.tensor([1,1])))  # 用受控断言验证关键不变量。

## 8. 可信制品与推理边界

线上接口只接收 `model_version`，不能让调用方传入任意模型冒充已发布 Conformer。制品绑定 state、architecture config、词表、blank、前端版本和训练特征快照；每次加载都重算 hash。hash 是完整性检查，不代替签名、对象存储 ACL 或回滚机制。

流式 Conformer 还需要 chunk mask、left context、卷积 cache 和端点检测。本例是离线全序列推理，不把它宣传为 streaming。

In [ ]:
def tensor_sha256(value):  # 定义本节可复用的核心函数。
    array = value.detach().cpu().contiguous().numpy()  # 计算并保存当前步骤的中间状态。
    digest = sha256(str(array.dtype).encode() + json.dumps(list(array.shape)).encode() + array.tobytes())  # 计算并保存当前步骤的中间状态。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

def module_sha256(module):  # 定义本节可复用的核心函数。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    for name, value in sorted(module.state_dict().items()):  # 遍历输入元素以累积或检查结果。
        digest.update(name.encode()); digest.update(tensor_sha256(value).encode())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

def json_sha25640(value):  # 定义本节可复用的核心函数。
    return sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()  # 返回当前分支计算出的结果。

def model_config40(model):  # 定义本节可复用的核心函数。
    return {"feature_dim": model.feature_dim, "vocab_size": model.vocab_size,  # 返回当前分支计算出的结果。
            "dim": model.dim, "heads": model.heads, "layers": model.layers_count}  # 执行当前语句以推进本节示例。

def bundle_sha25640(artifact):  # 定义本节可复用的核心函数。
    payload = {k: v for k, v in artifact.items() if k != "bundle_sha256"}  # 计算并保存当前步骤的中间状态。
    return sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()  # 返回当前分支计算出的结果。

artifact40 = {  # 计算并保存当前步骤的中间状态。
    "model_version": "conformer-ctc-demo-v1", "architecture": type(model40).__name__,  # 执行当前语句以推进本节示例。
    "config": model_config40(model40), "state_sha256": module_sha256(model40),  # 执行当前语句以推进本节示例。
    "vocab": VOCAB, "blank_id": BLANK_ID, "frontend_version": "synthetic-fbank-v1",  # 执行当前语句以推进本节示例。
    "code_version": "conformer-ctc-teaching-v1",  # 执行当前语句以推进本节示例。
    "train_features_sha256": tensor_sha256(train_features),  # 执行当前语句以推进本节示例。
    "train_lengths_sha256": tensor_sha256(train_lengths),  # 执行当前语句以推进本节示例。
    "train_targets_sha256": json_sha25640([target.tolist() for target in train_targets]),  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
artifact40["bundle_sha256"] = bundle_sha25640(artifact40)  # 计算并保存当前步骤的中间状态。
_REGISTRY40 = {artifact40["model_version"]: {"model": model40, "artifact": deepcopy(artifact40),  # 计算并保存当前步骤的中间状态。
                                             "train_features": train_features.clone(),  # 执行当前语句以推进本节示例。
                                             "train_lengths": train_lengths.clone(),  # 执行当前语句以推进本节示例。
                                             "train_targets": [target.clone() for target in train_targets]}}  # 执行当前语句以推进本节示例。

def load_trusted40(version):  # 定义本节可复用的核心函数。
    if version not in _REGISTRY40:  # 按当前条件选择后续控制路径。
        raise KeyError("unknown_model_version")  # 遇到非法合同立即显式失败。
    entry = _REGISTRY40[version]; model, artifact = entry["model"], entry["artifact"]  # 计算并保存当前步骤的中间状态。
    if type(model) is not ConformerCTC or artifact["architecture"] != type(model).__name__:  # 按当前条件选择后续控制路径。
        raise TypeError("architecture_mismatch")  # 遇到非法合同立即显式失败。
    if artifact["bundle_sha256"] != bundle_sha25640(artifact):  # 按当前条件选择后续控制路径。
        raise RuntimeError("bundle_mismatch")  # 遇到非法合同立即显式失败。
    if artifact["config"] != model_config40(model) or artifact["state_sha256"] != module_sha256(model):  # 按当前条件选择后续控制路径。
        raise RuntimeError("model_mismatch")  # 遇到非法合同立即显式失败。
    vocab = artifact.get("vocab", {})  # 计算并保存当前步骤的中间状态。
    blank_id = artifact.get("blank_id")  # 计算并保存当前步骤的中间状态。
    if artifact["config"]["vocab_size"] != len(vocab) or set(vocab) != set(range(len(vocab))):  # 按当前条件选择后续控制路径。
        raise RuntimeError("vocab_contract_mismatch")  # 遇到非法合同立即显式失败。
    if blank_id not in vocab or vocab[blank_id] != "<blank>" or blank_id != BLANK_ID:  # 按当前条件选择后续控制路径。
        raise RuntimeError("blank_contract_mismatch")  # 遇到非法合同立即显式失败。
    if artifact["train_features_sha256"] != tensor_sha256(entry["train_features"]):  # 按当前条件选择后续控制路径。
        raise RuntimeError("snapshot_mismatch")  # 遇到非法合同立即显式失败。
    if artifact["train_lengths_sha256"] != tensor_sha256(entry["train_lengths"]):  # 按当前条件选择后续控制路径。
        raise RuntimeError("snapshot_mismatch")  # 遇到非法合同立即显式失败。
    if artifact["train_targets_sha256"] != json_sha25640([target.tolist() for target in entry["train_targets"]]):  # 按当前条件选择后续控制路径。
        raise RuntimeError("snapshot_mismatch")  # 遇到非法合同立即显式失败。
    return model, artifact  # 返回当前分支计算出的结果。

@torch.no_grad()  # 为下方定义附加声明式配置。
def transcribe40(features, lengths, model_version="conformer-ctc-demo-v1"):  # 定义本节可复用的核心函数。
    model, artifact = load_trusted40(model_version)  # 计算并保存当前步骤的中间状态。
    model.eval()  # 执行当前语句以推进本节示例。
    log_probs, _, _ = model(features, lengths)  # 计算并保存当前步骤的中间状态。
    tokens = greedy_ctc_decode(log_probs, lengths, artifact["blank_id"])  # 计算并保存当前步骤的中间状态。
    return tokens, {"model_version": model_version, "bundle_sha256": artifact["bundle_sha256"],  # 返回当前分支计算出的结果。
                    "frontend_version": artifact["frontend_version"]}  # 执行当前语句以推进本节示例。

served_tokens40, trace40 = transcribe40(valid_features[:2], valid_lengths[:2])  # 计算并保存当前步骤的中间状态。
assert served_tokens40 == valid_predictions[:2]  # 用受控断言验证关键不变量。
assert trace40["bundle_sha256"] == artifact40["bundle_sha256"]  # 用受控断言验证关键不变量。

parameter40 = next(model40.parameters()); backup40 = parameter40.detach().clone()  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    with torch.no_grad(): parameter40.add_(0.1)  # 在受管理的上下文中执行操作。
    assert rejected40(lambda: transcribe40(valid_features[:1], valid_lengths[:1]))  # 用受控断言验证关键不变量。
finally:  # 无论结果如何都执行收尾逻辑。
    with torch.no_grad(): parameter40.copy_(backup40)  # 在受管理的上下文中执行操作。
target_backup40 = _REGISTRY40["conformer-ctc-demo-v1"]["train_targets"][0].clone()  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    _REGISTRY40["conformer-ctc-demo-v1"]["train_targets"][0][0] = 4  # 计算并保存当前步骤的中间状态。
    assert rejected40(lambda: transcribe40(valid_features[:1], valid_lengths[:1]))  # 用受控断言验证关键不变量。
finally:  # 无论结果如何都执行收尾逻辑。
    _REGISTRY40["conformer-ctc-demo-v1"]["train_targets"][0] = target_backup40  # 计算并保存当前步骤的中间状态。
artifact_backup40 = deepcopy(_REGISTRY40["conformer-ctc-demo-v1"]["artifact"])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    _REGISTRY40["conformer-ctc-demo-v1"]["artifact"]["blank_id"] = 4  # 计算并保存当前步骤的中间状态。
    changed_artifact40 = _REGISTRY40["conformer-ctc-demo-v1"]["artifact"]  # 计算并保存当前步骤的中间状态。
    changed_artifact40["bundle_sha256"] = bundle_sha25640(changed_artifact40)  # 计算并保存当前步骤的中间状态。
    assert rejected40(lambda: transcribe40(valid_features[:1], valid_lengths[:1]))  # 用受控断言验证关键不变量。
finally:  # 无论结果如何都执行收尾逻辑。
    _REGISTRY40["conformer-ctc-demo-v1"]["artifact"] = artifact_backup40  # 计算并保存当前步骤的中间状态。
assert transcribe40(valid_features[:1], valid_lengths[:1])[0][0] == valid_predictions[0]  # 用受控断言验证关键不变量。

## 9. 面试总结与研究来源

完整回答应区分三层：Conformer 负责上下文表征，CTC 对所有单调 alignment 求和，decoder/语言模型负责进一步搜索约束。greedy decode 不是 beam search；离线 mask 也不是流式 cache。

- Gulati et al., *Conformer: Convolution-augmented Transformer for Speech Recognition*：https://arxiv.org/abs/2005.08100
- Graves et al., *Connectionist Temporal Classification*：https://www.cs.toronto.edu/~graves/icml_2006.pdf
- PyTorch `nn.Module` 文档：https://pytorch.org/docs/stable/generated/torch.nn.Module.html

生产系统还要补真实声学前端、SpecAugment、subsampling、beam search/LM fusion、CER/WER、流式状态、说话人切分、噪声与公平性评估。

In [ ]:
assert type(model40).__name__ == "ConformerCTC"  # 用受控断言验证关键不变量。
assert len(model40.blocks) == artifact40["config"]["layers"]  # 用受控断言验证关键不变量。
assert model40.blocks[0].convolution.depthwise.groups == model40.dim  # 用受控断言验证关键不变量。
assert losses40[-1] < losses40[0]  # 用受控断言验证关键不变量。
assert sequence_accuracy40 >= 0.75  # 用受控断言验证关键不变量。
assert bundle_sha25640(artifact40) == artifact40["bundle_sha256"]  # 用受控断言验证关键不变量。
assert trace40["frontend_version"] == "synthetic-fbank-v1"  # 用受控断言验证关键不变量。
assert model40.training is False  # 用受控断言验证关键不变量。
print("Conformer、手写 CTC、padding 与可信制品回归全部通过。")  # 执行当前语句以推进本节示例。